# 2_Modelo_Preditivo

Objetivo: prever **risco de defasagem no ano seguinte** com foco em alta sensibilidade (Recall).

## 1) Setup

In [ ]:
import re
import unicodedata
from pathlib import Path
import warnings

import pickle
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
px.defaults.template = "plotly_white"

def detect_project_root(start=None):
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "app.py").exists():
            return candidate
    return current

PROJECT_ROOT = detect_project_root()
DATA_PATH = PROJECT_ROOT / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "pede_consolidado_2022_2024.csv"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET_IAN_THRESHOLD = 5.0
TARGET_IDA_THRESHOLD = 6.0
RECALL_OBJECTIVE = 0.80

## 2) Funcoes de carga e harmonizacao (fallback caso CSV consolidado nao exista)

In [ ]:
def normalize_column_name(col_name: str) -> str:
    normalized = unicodedata.normalize("NFKD", str(col_name)).encode("ascii", "ignore").decode("ascii")
    normalized = re.sub(r"[^0-9a-zA-Z]+", "_", normalized).strip("_").lower()
    return normalized


def clean_empty_strings(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.strip()
    return cleaned.replace({"": np.nan, "nan": np.nan, "None": np.nan, "<NA>": np.nan})


def coalesce_columns(df: pd.DataFrame, new_col: str, candidates: list[str], numeric: bool = False) -> None:
    result = None
    for col in candidates:
        if col not in df.columns:
            continue
        values = pd.to_numeric(df[col], errors="coerce") if numeric else clean_empty_strings(df[col])
        result = values if result is None else result.where(result.notna(), values)
    if result is None:
        result = pd.Series(np.nan, index=df.index)
    df[new_col] = result


def standardize_gender(value: str) -> str:
    if pd.isna(value):
        return np.nan
    value_clean = str(value).strip().lower()
    mapping = {"menino": "Masculino", "masculino": "Masculino", "menina": "Feminino", "feminino": "Feminino"}
    return mapping.get(value_clean, str(value).strip())


def standardize_stone(value: str) -> str:
    if pd.isna(value):
        return np.nan
    value_norm = unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii")
    value_norm = value_norm.strip().lower()
    mapping = {
        "quartzo": "Quartzo",
        "agata": "Agata",
        "ametista": "Ametista",
        "topazio": "Topazio",
        "incluir": np.nan,
    }
    return mapping.get(value_norm, str(value).strip())


def extract_phase_code(value: str) -> str:
    if pd.isna(value):
        return np.nan
    raw = str(value).strip().upper()
    if raw in {"", "NAN", "NONE"}:
        return np.nan
    if "ALFA" in raw:
        return "ALFA"
    match = re.search(r"FASE\s*([0-9]+)", raw)
    if match:
        return f"FASE {int(match.group(1))}"
    if re.fullmatch(r"[0-9]+", raw):
        phase_num = int(raw)
        return "ALFA" if phase_num == 0 else f"FASE {phase_num}"
    return np.nan


def load_and_prepare_data(data_path: Path) -> pd.DataFrame:
    xls = pd.ExcelFile(data_path)
    all_frames = []

    for sheet in xls.sheet_names:
        year_match = re.search(r"20\d{2}", sheet)
        if not year_match:
            continue
        year = int(year_match.group())

        raw = xls.parse(sheet)
        raw = raw.dropna(how="all").copy()
        raw.columns = [normalize_column_name(c) for c in raw.columns]

        if "ra" not in raw.columns:
            continue
        raw["ra"] = clean_empty_strings(raw["ra"])
        raw = raw[~raw["ra"].str.upper().isin(["RA-1", "NAN"])]
        raw = raw[raw["ra"].notna()]
        raw["ano_referencia"] = year

        coalesce_columns(raw, "idade", ["idade", "idade_22"], numeric=True)
        coalesce_columns(raw, "instituicao_ensino", ["instituicao_de_ensino", "escola"], numeric=False)
        coalesce_columns(raw, "fase_ideal", ["fase_ideal", "fase_ideal_"], numeric=False)
        coalesce_columns(raw, "defasagem", ["defasagem", "defas"], numeric=True)
        coalesce_columns(raw, "nota_matematica", ["mat", "matem"], numeric=True)
        coalesce_columns(raw, "nota_portugues", ["por", "portug"], numeric=True)
        coalesce_columns(raw, "nota_ingles", ["ing", "ingles"], numeric=True)
        coalesce_columns(raw, "inde_ano", ["inde_2024", "inde_2023", "inde_23", "inde_22"], numeric=True)
        coalesce_columns(raw, "pedra_ano", ["pedra_2024", "pedra_2023", "pedra_23", "pedra_22"], numeric=False)
        coalesce_columns(raw, "ativo_inativo", ["ativo_inativo", "ativo_inativo_1"], numeric=False)

        for col in ["fase", "turma", "genero", "ian", "ida", "ieg", "iaa", "ips", "ipp", "ipv"]:
            if col not in raw.columns:
                raw[col] = np.nan

        phase_from_fase = raw["fase"].apply(extract_phase_code)
        phase_from_ideal = raw["fase_ideal"].apply(extract_phase_code)
        raw["fase_programa"] = phase_from_fase.where(phase_from_fase.notna(), phase_from_ideal)

        for num_col in [
            "idade",
            "defasagem",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]:
            raw[num_col] = pd.to_numeric(raw[num_col], errors="coerce")

        for score_col in [
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]:
            raw[score_col] = raw[score_col].clip(lower=0, upper=10)

        raw["genero"] = raw["genero"].apply(standardize_gender)
        raw["pedra_ano"] = raw["pedra_ano"].apply(standardize_stone)

        keep_cols = [
            "ra",
            "ano_referencia",
            "idade",
            "genero",
            "instituicao_ensino",
            "fase_programa",
            "turma",
            "pedra_ano",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "defasagem",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
            "ativo_inativo",
        ]
        for col in keep_cols:
            if col not in raw.columns:
                raw[col] = np.nan

        all_frames.append(raw[keep_cols].copy())

    df = pd.concat(all_frames, ignore_index=True)
    df = df.drop_duplicates(subset=["ra", "ano_referencia"], keep="first")
    df = df.sort_values(["ra", "ano_referencia"]).reset_index(drop=True)

    for cat_col in ["genero", "instituicao_ensino", "fase_programa", "turma", "pedra_ano", "ativo_inativo"]:
        df[cat_col] = clean_empty_strings(df[cat_col])

    return df


def load_consolidated_or_raw() -> pd.DataFrame:
    if PROCESSED_PATH.exists():
        df = pd.read_csv(PROCESSED_PATH)
        for num_col in [
            "idade",
            "defasagem",
            "inde_ano",
            "ian",
            "ida",
            "ieg",
            "iaa",
            "ips",
            "ipp",
            "ipv",
            "nota_matematica",
            "nota_portugues",
            "nota_ingles",
        ]:
            if num_col in df.columns:
                df[num_col] = pd.to_numeric(df[num_col], errors="coerce")
        return df
    return load_and_prepare_data(DATA_PATH)

## 3) Engenharia de target: risco de defasagem no ano subsequente

Definicao usada (ajustavel):

`risco_defasagem_t1 = 1 se (IAN_t+1 <= 5.0) E (IDA_t+1 <= 6.0)`

In [ ]:
df = load_consolidated_or_raw().copy()

expected_cols = [
    "ra",
    "ano_referencia",
    "idade",
    "genero",
    "instituicao_ensino",
    "fase_programa",
    "turma",
    "pedra_ano",
    "inde_ano",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "defasagem",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
    "ativo_inativo",
]
for col in expected_cols:
    if col not in df.columns:
        df[col] = np.nan

base = df.rename(columns={"ano_referencia": "ano_base"}).copy()
future = df[["ra", "ano_referencia", "ian", "ida"]].copy()
future = future.rename(columns={"ano_referencia": "ano_base", "ian": "ian_prox", "ida": "ida_prox"})
future["ano_base"] = future["ano_base"] - 1

model_df = base.merge(future, on=["ra", "ano_base"], how="left")
model_df["target_disponivel"] = model_df["ian_prox"].notna() & model_df["ida_prox"].notna()
model_df["risco_defasagem_t1"] = (
    (model_df["ian_prox"] <= TARGET_IAN_THRESHOLD) & (model_df["ida_prox"] <= TARGET_IDA_THRESHOLD)
).astype("Int64")

labeled_df = model_df[(model_df["target_disponivel"]) & (model_df["ano_base"].isin([2022, 2023]))].copy()

print("Base total:", df.shape)
print("Base rotulada para modelagem:", labeled_df.shape)
display(labeled_df[["ano_base", "risco_defasagem_t1"]].value_counts().rename("contagem").reset_index())

class_balance = labeled_df.groupby("ano_base", as_index=False)["risco_defasagem_t1"].mean().rename(columns={"risco_defasagem_t1": "taxa_risco"})
class_balance["taxa_risco"] = class_balance["taxa_risco"] * 100
display(class_balance.round(2))

fig_balance = px.bar(
    class_balance,
    x="ano_base",
    y="taxa_risco",
    text_auto=".1f",
    title="Taxa de risco no target por ano-base",
    labels={"ano_base": "Ano-base", "taxa_risco": "Taxa de risco (%)"},
)
fig_balance.show()

## 4) Split temporal, preprocessamento e tratamento de desbalanceamento

Estrategia temporal:
- treino: ano-base 2022 (prevendo 2023)
- teste: ano-base 2023 (prevendo 2024)

In [ ]:
numeric_features = [
    "idade",
    "inde_ano",
    "ian",
    "ida",
    "ieg",
    "iaa",
    "ips",
    "ipp",
    "ipv",
    "defasagem",
    "nota_matematica",
    "nota_portugues",
    "nota_ingles",
]
categorical_features = ["genero", "instituicao_ensino", "fase_programa", "turma", "pedra_ano", "ativo_inativo"]
feature_columns = numeric_features + categorical_features

for col in feature_columns:
    if col not in labeled_df.columns:
        labeled_df[col] = np.nan

train_df = labeled_df[labeled_df["ano_base"] == 2022].copy()
test_df = labeled_df[labeled_df["ano_base"] == 2023].copy()
if train_df.empty or test_df.empty:
    raise ValueError("Split temporal invalido. Verifique os dados por ano-base.")

X_train = train_df[feature_columns].copy()
y_train = train_df["risco_defasagem_t1"].astype(int)
X_test = test_df[feature_columns].copy()
y_test = test_df["risco_defasagem_t1"].astype(int)


def random_oversample(X: pd.DataFrame, y: pd.Series, random_state: int = 42) -> tuple[pd.DataFrame, pd.Series]:
    """Oversampling simples da classe minoritaria."""
    y = y.reset_index(drop=True)
    X = X.reset_index(drop=True)
    class_counts = y.value_counts()
    if len(class_counts) < 2:
        return X, y
    majority_class = class_counts.idxmax()
    minority_class = class_counts.idxmin()
    majority_count = class_counts.max()
    minority_count = class_counts.min()
    if minority_count / majority_count >= 0.70:
        return X, y
    rng = np.random.default_rng(random_state)
    minority_idx = np.where(y == minority_class)[0]
    extra_idx = rng.choice(minority_idx, size=majority_count - minority_count, replace=True)
    X_bal = pd.concat([X, X.iloc[extra_idx]], ignore_index=True)
    y_bal = pd.concat([y, y.iloc[extra_idx]], ignore_index=True)
    shuffled_idx = rng.permutation(len(y_bal))
    return X_bal.iloc[shuffled_idx].reset_index(drop=True), y_bal.iloc[shuffled_idx].reset_index(drop=True)


X_train_bal, y_train_bal = random_oversample(X_train, y_train, random_state=RANDOM_STATE)
print("Treino original:", X_train.shape, "| taxa positiva:", round(y_train.mean(), 3))
print("Treino balanceado:", X_train_bal.shape, "| taxa positiva:", round(y_train_bal.mean(), 3))
print("Teste:", X_test.shape, "| taxa positiva:", round(y_test.mean(), 3))

numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]
)
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

## 5) Treino e comparacao de modelos

Modelos:
- Regressao Logistica (baseline interpretavel)
- Random Forest (maior capacidade preditiva)

A otimizacao prioriza Recall via ajuste de threshold.

In [ ]:
models = {
    "Regressao Logistica": LogisticRegression(
        max_iter=4000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=500,
        max_depth=10,
        min_samples_leaf=4,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}


def choose_threshold_for_recall(y_true: pd.Series, y_prob: np.ndarray, min_recall: float = 0.80) -> float:
    """Seleciona threshold com recall minimo e melhor precisao dentre os validos."""
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    recall = recall[:-1]
    precision = precision[:-1]
    valid_idx = np.where(recall >= min_recall)[0]
    if len(valid_idx) == 0:
        return 0.50
    best_local_idx = valid_idx[np.argmax(precision[valid_idx])]
    return float(thresholds[best_local_idx])


def evaluate_probabilities(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "y_pred": y_pred,
    }


results = {}
for model_name, estimator in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("model", estimator),
        ]
    )
    pipeline.fit(X_train_bal, y_train_bal)
    train_prob = pipeline.predict_proba(X_train)[:, 1]
    threshold = choose_threshold_for_recall(y_train, train_prob, min_recall=RECALL_OBJECTIVE)
    test_prob = pipeline.predict_proba(X_test)[:, 1]
    metrics = evaluate_probabilities(y_test, test_prob, threshold)
    results[model_name] = {
        "pipeline": pipeline,
        "threshold": threshold,
        "metrics": metrics,
        "test_prob": test_prob,
    }

metrics_table = pd.DataFrame(
    [
        {
            "modelo": name,
            "threshold": result["threshold"],
            "accuracy": result["metrics"]["accuracy"],
            "precision": result["metrics"]["precision"],
            "recall": result["metrics"]["recall"],
            "f1": result["metrics"]["f1"],
            "roc_auc": result["metrics"]["roc_auc"],
        }
        for name, result in results.items()
    ]
).sort_values(["recall", "roc_auc"], ascending=False)
display(metrics_table.round(4))

## 6) Avaliacao: matriz de confusao e ROC-AUC

In [ ]:
for model_name, result in results.items():
    y_pred = result["metrics"]["y_pred"]
    cm = confusion_matrix(y_test, y_pred)
    fig_cm = go.Figure(
        data=go.Heatmap(
            z=cm,
            x=["Pred 0", "Pred 1"],
            y=["Real 0", "Real 1"],
            text=cm,
            texttemplate="%{text}",
            colorscale="Blues",
        )
    )
    fig_cm.update_layout(title=f"Matriz de confusao - {model_name}")
    fig_cm.show()

roc_fig = go.Figure()
for model_name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result["test_prob"])
    roc_fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{model_name} (AUC={result['metrics']['roc_auc']:.3f})",
        )
    )
roc_fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="Aleatorio", line=dict(dash="dash")))
roc_fig.update_layout(
    title="Curva ROC - Comparacao de modelos",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    yaxis=dict(scaleanchor="x", scaleratio=1),
    xaxis=dict(constrain="domain"),
)
roc_fig.show()

## 7) Explicabilidade: Feature Importance (e SHAP opcional)

In [ ]:
best_model_name = max(results.keys(), key=lambda name: (results[name]["metrics"]["recall"], results[name]["metrics"]["roc_auc"]))
best_pipeline = results[best_model_name]["pipeline"]
best_threshold = results[best_model_name]["threshold"]

print("Melhor modelo:", best_model_name)
print("Threshold otimizado:", round(best_threshold, 4))

fitted_preprocessor = best_pipeline.named_steps["preprocessor"]
fitted_model = best_pipeline.named_steps["model"]
feature_names = fitted_preprocessor.get_feature_names_out()

if hasattr(fitted_model, "feature_importances_"):
    importances = fitted_model.feature_importances_
    importance_df = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
else:
    coef = np.abs(fitted_model.coef_[0])
    importance_df = pd.DataFrame({"feature": feature_names, "importance": coef}).sort_values("importance", ascending=False)

display(importance_df.head(25))

fig_importance = px.bar(
    importance_df.head(20).sort_values("importance", ascending=True),
    x="importance",
    y="feature",
    orientation="h",
    title=f"Top 20 fatores mais relevantes - {best_model_name}",
    labels={"importance": "Importancia", "feature": "Feature"},
)
fig_importance.show()

try:
    import shap

    X_test_sample = X_test.sample(min(250, len(X_test)), random_state=RANDOM_STATE)
    X_test_transformed = fitted_preprocessor.transform(X_test_sample)
    if hasattr(X_test_transformed, "toarray"):
        X_test_transformed = X_test_transformed.toarray()

    explainer = shap.Explainer(fitted_model)
    shap_values = explainer(X_test_transformed)
    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
    shap_df = pd.DataFrame({"feature": feature_names, "mean_abs_shap": mean_abs_shap}).sort_values("mean_abs_shap", ascending=False)
    display(shap_df.head(15))

    fig_shap = px.bar(
        shap_df.head(15).sort_values("mean_abs_shap", ascending=True),
        x="mean_abs_shap",
        y="feature",
        orientation="h",
        title="Top 15 features por SHAP medio absoluto",
        labels={"mean_abs_shap": "|SHAP| medio", "feature": "Feature"},
    )
    fig_shap.show()
except Exception as shap_error:
    print("SHAP nao executado (usando apenas Feature Importance):", shap_error)

## 8) Exportacao de artefatos (modelo, scaler e score 2024)

In [ ]:
default_numeric = {col: float(labeled_df[col].median()) if labeled_df[col].notna().any() else 0.0 for col in numeric_features}
default_categorical = {}
for col in categorical_features:
    mode_series = labeled_df[col].dropna().astype(str)
    default_categorical[col] = mode_series.mode().iloc[0] if not mode_series.empty else "Nao Informado"

category_options = {col: sorted(labeled_df[col].dropna().astype(str).unique().tolist()) for col in categorical_features}
training_info = {
    "train_years": [2022],
    "test_years": [2023],
    "train_rows": int(len(train_df)),
    "test_rows": int(len(test_df)),
    "target_positive_rate_train": float(y_train.mean()),
    "target_positive_rate_test": float(y_test.mean()),
}

bundle = {
    "model_name": best_model_name,
    "pipeline": best_pipeline,
    "threshold": float(best_threshold),
    "feature_columns": feature_columns,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "target_rule": {
        "description": "risco=1 se (IAN_t+1 <= 5.0) e (IDA_t+1 <= 6.0)",
        "ian_threshold": TARGET_IAN_THRESHOLD,
        "ida_threshold": TARGET_IDA_THRESHOLD,
    },
    "healthy_thresholds": {"ian": 5.0, "ida": 6.0, "ieg": 7.0, "ips": 7.0, "ipp": 7.0, "ipv": 7.0},
    "default_inputs": {**default_numeric, **default_categorical},
    "category_options": category_options,
    "training_info": training_info,
}

model_path = ARTIFACT_DIR / "modelo_risco_defasagem.pkl"
with model_path.open("wb") as file:
    pickle.dump(bundle, file)

scaler_obj = best_pipeline.named_steps["preprocessor"].named_transformers_["num"].named_steps["scaler"]
scaler_path = ARTIFACT_DIR / "scaler_numerico.pkl"
with scaler_path.open("wb") as file:
    pickle.dump(scaler_obj, file)

importance_path = ARTIFACT_DIR / "feature_importance.csv"
importance_df.to_csv(importance_path, index=False)

score_2024 = model_df[model_df["ano_base"] == 2024].copy()
if not score_2024.empty:
    X_2024 = score_2024[feature_columns].copy()
    score_2024["prob_risco"] = best_pipeline.predict_proba(X_2024)[:, 1]
    score_2024["risco_predito"] = (score_2024["prob_risco"] >= best_threshold).astype(int)
    score_2024["nivel_risco"] = pd.cut(
        score_2024["prob_risco"],
        bins=[-0.01, 0.33, 0.66, 1.0],
        labels=["Baixo", "Moderado", "Alto"],
    )
    score_2024_out = ARTIFACT_DIR / "predicoes_risco_2024.csv"
    score_2024[["ra", "ano_base", "prob_risco", "risco_predito", "nivel_risco"]].to_csv(score_2024_out, index=False)

print("Artefatos exportados:")
print("-", model_path.resolve())
print("-", scaler_path.resolve())
print("-", importance_path.resolve())
if not score_2024.empty:
    print("-", score_2024_out.resolve())